# Payments Operations Analysis

**Section 4e Demo** — Notebooks combine SQL and Python in a single governed environment.
This notebook queries payment data, analyses failure patterns, and produces operational charts.

Run inside Snowflake — no data leaves, uses your warehouse compute.

In [ ]:
USE DATABASE BARCLAYS_DEMO;
USE SCHEMA RAW;
USE WAREHOUSE BARCLAYS_DEMO_WH;

## 1. SQL: Payment Volume by Type and Status
Standard SQL cell — same queries you'd run in a worksheet.

In [ ]:
SELECT 
    PAYMENT_TYPE,
    STATUS,
    COUNT(*) AS TXN_COUNT,
    ROUND(SUM(AMOUNT), 0) AS TOTAL_VALUE,
    ROUND(AVG(PROCESSING_TIME_MS), 0) AS AVG_PROCESSING_MS
FROM PAYMENTS
GROUP BY PAYMENT_TYPE, STATUS
ORDER BY PAYMENT_TYPE, TXN_COUNT DESC

## 2. Python: Failure Rate Analysis
Convert SQL results to Pandas and compute failure rates per payment type.

In [ ]:
import pandas as pd

df = payment_volume.to_pandas()

pivot = df.pivot_table(index='PAYMENT_TYPE', columns='STATUS', values='TXN_COUNT', fill_value=0).reset_index()
pivot['TOTAL'] = pivot.sum(axis=1, numeric_only=True)
pivot['FAILURE_RATE'] = round(pivot.get('FAILED', 0) / pivot['TOTAL'] * 100, 2)
pivot['SUCCESS_RATE'] = round(pivot.get('COMPLETED', 0) / pivot['TOTAL'] * 100, 2)

print(pivot[['PAYMENT_TYPE', 'TOTAL', 'FAILURE_RATE', 'SUCCESS_RATE']].to_string(index=False))

## 3. Visualisation: Daily Failure Trends
SQL query feeds directly into a Python chart — no data export needed.

In [ ]:
SELECT 
    PAYMENT_DATE::DATE AS DAY,
    PAYMENT_TYPE,
    COUNT(*) AS TOTAL_TXNS,
    COUNT(CASE WHEN STATUS = 'FAILED' THEN 1 END) AS FAILED_TXNS,
    ROUND(100.0 * COUNT(CASE WHEN STATUS = 'FAILED' THEN 1 END) / COUNT(*), 2) AS FAIL_RATE_PCT
FROM PAYMENTS
WHERE PAYMENT_DATE >= DATEADD(DAY, -30, (SELECT MAX(PAYMENT_DATE) FROM PAYMENTS))
GROUP BY DAY, PAYMENT_TYPE
ORDER BY DAY

In [ ]:
import altair as alt

df_daily = daily_metrics.to_pandas()

daily_agg = df_daily.groupby('DAY', as_index=False).agg(
    TOTAL_TXNS=('TOTAL_TXNS', 'sum'),
    FAILED_TXNS=('FAILED_TXNS', 'sum')
)
daily_agg['FAIL_RATE'] = round(daily_agg['FAILED_TXNS'] / daily_agg['TOTAL_TXNS'] * 100, 2)

chart = alt.Chart(daily_agg).mark_line(
    point=True, color='#ef4444'
).encode(
    x=alt.X('DAY:T', title='Date'),
    y=alt.Y('FAIL_RATE:Q', title='Failure Rate %', scale=alt.Scale(zero=True)),
    tooltip=['DAY:T', 'FAIL_RATE:Q', 'TOTAL_TXNS:Q']
).properties(
    title='Daily Payment Failure Rate (Last 30 Days)',
    width=700, height=300
)

chart

## 4. Breakdown by Payment Type
Compare failure rates across SWIFT, BACS, CHAPS, etc.

In [ ]:
type_agg = df_daily.groupby('PAYMENT_TYPE', as_index=False).agg(
    TOTAL=('TOTAL_TXNS', 'sum'),
    FAILED=('FAILED_TXNS', 'sum')
)
type_agg['FAIL_RATE'] = round(type_agg['FAILED'] / type_agg['TOTAL'] * 100, 2)

bar = alt.Chart(type_agg).mark_bar(
    cornerRadiusTopLeft=4, cornerRadiusTopRight=4
).encode(
    x=alt.X('PAYMENT_TYPE:N', title='Payment Type', sort='-y'),
    y=alt.Y('FAIL_RATE:Q', title='Failure Rate %'),
    color=alt.condition(
        alt.datum.FAIL_RATE > 15,
        alt.value('#ef4444'),
        alt.value('#29B5E8')
    ),
    tooltip=['PAYMENT_TYPE:N', 'FAIL_RATE:Q', 'TOTAL:Q', 'FAILED:Q']
).properties(
    title='Failure Rate by Payment Type',
    width=600, height=300
)

bar

## 5. Processing Time Distribution
Are any payment types consistently slow?

In [ ]:
SELECT 
    PAYMENT_TYPE,
    ROUND(AVG(PROCESSING_TIME_MS), 0) AS AVG_MS,
    ROUND(PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY PROCESSING_TIME_MS), 0) AS P50_MS,
    ROUND(PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY PROCESSING_TIME_MS), 0) AS P95_MS,
    ROUND(PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY PROCESSING_TIME_MS), 0) AS P99_MS,
    MAX(PROCESSING_TIME_MS) AS MAX_MS
FROM PAYMENTS
GROUP BY PAYMENT_TYPE
ORDER BY P95_MS DESC

In [ ]:
df_proc = processing_times.to_pandas()

melted = df_proc.melt(
    id_vars='PAYMENT_TYPE', 
    value_vars=['P50_MS', 'P95_MS'],
    var_name='Percentile', value_name='Milliseconds'
)

grouped = alt.Chart(melted).mark_bar(
    cornerRadiusTopLeft=3, cornerRadiusTopRight=3
).encode(
    x=alt.X('PAYMENT_TYPE:N', title='Payment Type'),
    y=alt.Y('Milliseconds:Q', title='Processing Time (ms)'),
    color=alt.Color('Percentile:N', 
        scale=alt.Scale(range=['#29B5E8', '#f59e0b']),
        legend=alt.Legend(title='Metric')
    ),
    xOffset='Percentile:N',
    tooltip=['PAYMENT_TYPE:N', 'Percentile:N', 'Milliseconds:Q']
).properties(
    title='Processing Time: P50 vs P95 by Payment Type',
    width=600, height=300
)

grouped

## Summary

This notebook demonstrates:
- **SQL cells** querying Snowflake tables directly
- **Python cells** transforming results with Pandas
- **Altair charts** for operational visualisation
- All running on **Snowflake compute** — no data leaves the platform

### Deployment
To schedule this notebook as a recurring pipeline:
1. Click **Schedule** in the notebook toolbar
2. Set frequency (e.g. daily at 06:00)
3. Assign a warehouse

The notebook becomes a governed, scheduled job — same as a Task or dbt run.